# 02 - Annotate atlas / perturbed cells using milestone annotations from control cells

In [ ]:
%autosave 0

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import snapseed as snap
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import marsilea as ma
import marsilea.plotter as mp
import cellmapper
from dynaconf import Dynaconf

# Global Scanpy settings
sc.settings.verbosity = 2               # Show progress
sc.settings.set_figure_params(
    dpi=150,                            # High-res output
    dpi_save=300,                       # High-res when saving
    format='pdf',                       # or 'pdf', 'svg'
    facecolor='white',                  # or 'none' for transparent
    frameon=False,                      # No outer frame
    vector_friendly=True,               # No rasterization warnings
    fontsize=10,                        # Base font size
    figsize=(4, 4),                     # Default figure size
    transparent=True                    # Transparent background if saving
)

mpl.rcParams.update({
    "svg.fonttype": "none",       
    "pdf.fonttype": 42,           
    "legend.fontsize": 6,
    "axes.titlesize": 6,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
})

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Load data 

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_eec.h5ad")

In [ ]:
adata.obs["cystic"].value_counts()

In [ ]:
adata_query = adata[~adata.obs["condition"].isin(["Control"])]
adata_ref = adata[adata.obs["condition"].isin(["Control"])]

## Annotate using controls

In [ ]:
adata_controls = sc.read(anndata_dir / "tf_ko_panel_control_mapped_vasa_milestones_including_cystic_v3.h5ad")

In [ ]:
adata_controls.obs

In [ ]:
for col in ["cell_type", "annotation_pred", "milestones_pred", "milestones_conf"]:
    adata_ref.obs[col] = adata_controls[list(adata_ref.obs.index)].obs[col].values

In [ ]:
sc.pl.embedding(adata_ref, basis="X_umap_shared_rep", color=["cell_type", "annotation_pred", "milestones_pred"], legend_loc="on data")

In [ ]:
cmap = cellmapper.CellMapper(adata_query, adata_ref).map(
    obs_keys=["cell_type", "annotation_pred", "milestones_pred"], use_rep="shared_rep", knn_method="pynndescent"
)
cmap

In [ ]:
sc.pl.embedding(adata_query, basis="X_umap_shared_rep", color=["annotation_pred_pred", "milestones_pred_pred"], legend_loc="on data")

In [ ]:
# Save the annotated query adata
adata_query.write(anndata_dir / "tf_ko_panel_contrastiveVI_perturbed_annotated_eecs_milestones_including_cystic_v3.h5ad")

# Save the annotated reference adata
adata_ref.write(anndata_dir / "tf_ko_panel_contrastiveVI_controls_annotated_eecs_milestones_including_cystic_v3.h5ad")

In [ ]:
# Load the objects again
adata_query = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_perturbed_annotated_eecs_milestones_including_cystic_v3.h5ad")
adata_ref = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_controls_annotated_eecs_milestones_including_cystic_v3.h5ad")

In [ ]:
for col in ["cell_type", "annotation_pred", "milestones_pred", "milestones_conf"]:
    adata_ref.obs[col] = adata_controls[list(adata_ref.obs.index)].obs[col].values

In [ ]:
adata_query.obs["annotation_pred"] = adata_query.obs["annotation_pred_pred"].values
adata_query.obs["milestones_pred"] = adata_query.obs["milestones_pred_pred"].values
adata_ref.obs["milestones_pred_conf"] = adata_ref.obs["milestones_conf"].values

In [ ]:
adata_query = adata_query.raw.to_adata()
adata_ref = adata_ref.raw.to_adata()

In [ ]:
adata_combined = adata_ref.concatenate(adata_query, batch_key="dataset", batch_categories=["Control", "Perturbed"], index_unique=None)

In [ ]:
# Load mapping
mapping = dict(zip(["100", "140", "186", "120", "59", "170", "165", "17"], 
                   ["X cells","D cells","X/D/I/K cells","EC/X/D/I/K cells",
                    "X/D/I/K cells","NGN3+ cells", "EC cells", "I/K cells"]))

In [ ]:
adata_combined.obs["milestones_pred"] = adata_combined.obs["milestones_pred"].map(mapping)

In [ ]:
adata_combined.write(anndata_dir / "tf_ko_panel_contrastiveVI_annotated_eecs_milestones_including_cystic_v3.h5ad")

In [ ]:
adata_combined = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_annotated_eecs_milestones_including_cystic_v3.h5ad")

In [ ]:
adata_combined

In [ ]:
adata_combined = adata_combined[adata_combined.obs["cystic"] == "cystic"]

In [ ]:
adata_combined.write(anndata_dir / "tf_ko_panel_contrastiveVI_annotated_eecs_milestones_only_cystic_v3.h5ad")

In [ ]:
adata_combined = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_annotated_eecs_milestones_including_cystic.h5ad")

In [ ]:
# Get average number of cells for each milestone
adata_combined.obs.groupby(["sample", "milestones_pred"]).size().unstack(fill_value=0)

In [ ]:
adata_combined = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_annotated_eecs_milestones_including_cystic_v3.h5ad")

In [ ]:
# Get average number of cells for each milestone
adata_combined.obs.groupby(["sample", "milestones_pred"]).size().unstack(fill_value=0)

In [ ]:
annotation_palette = {
    "eecs_milestones_pred": {
        # Pure populations
        "NGN3+ cells": "#9467bd",   # purple (progenitors)
        "EC/X/D/I/K cells": "#fdae61",    # warm orange blend
        "EC cells": "#d62728",      # strong red
        "X/D/I/K cells": "#c2a5cf",       # lavender blend
        "X cells": "#e377c2",       # magenta
        "D cells": "#17becf",       # cyan
        "I/K cells": "#bcbd22",     # olive
    },
    "eecs_annotation_pred": {
        # EEC progenitors
        "EEC Progenitors": "#9467bd",   # purple

        # EEC subtypes 
        "EC Cells": "#d62728",            # strong red
        "X Cells": "#e377c2",             # magenta
        "D Cells": "#17becf",             # cyan / turquoise
        "I/N Cells": "#7f7f7f",           # dark grey
        "K Cells": "#bcbd22",             # olive
    }
}

In [ ]:
adata_combined

In [ ]:
adata_controls = adata_combined[adata_combined.obs["condition"].isin(["Control"])]

In [ ]:
sc.pl.embedding(adata_controls, basis="X_umap_shared_rep", color="annotation", palette=annotation_palette["eecs_annotation_pred"], legend_loc="on data", size=5, save="_parse_mapped_vasa_annotation.pdf")

In [ ]:
sc.pl.embedding(adata_controls, basis="X_umap_shared_rep", color="milestones_pred", palette=annotation_palette["eecs_milestones_pred"], legend_loc="on data", size=5, save="_parse_mapped_vasa_milestones.pdf")

In [ ]:
sc.pl.embedding(adata_controls, basis="X_umap_shared_rep", color="milestones_pred_conf", cmap="inferno", legend_loc="on data", size=20)#, save="_parse_mapped_vasa_milestones_conf.pdf")